<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/mean_escape_time/double_well/double_well_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Double-well mean escape time: five-method comparison

This notebook compares five methods for the double-well mean exit time in an ellipse. Read the problem in Section 1, then edit the labeled problem, model, sampling, and training cells in Sections 2–5. Run top to bottom: Section 6 creates a fresh run, saves its seeds and numerical data, and trains the selected methods; Section 7 evaluates the saved checkpoints and creates the complete run archive. Drive is optional, and the notebook passes its output folder between cells.


## 1. Problem

Let

$$\Phi(x,y)=A\left(1-e^{-((x-1)^2+(y-1/2)^2)/\nu^2}\right)
\left(1-e^{-((x+1)^2+(y+1/2)^2)/\nu^2}\right),$$

$b=-\nabla\Phi$, and

$$dX_t=b(X_t)\,dt+\begin{pmatrix}\sqrt{2\epsilon}&0\\0&\sqrt{\epsilon}\end{pmatrix}dB_t.$$

The safe domain is the ellipse $\Omega=\{(x,y):(x/a)^2+(y/b)^2<1\}$ with $a=2$ and $b=1.5$. The mean exit time solves

$$-\mathcal L\tau=1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

$$\mathcal L\tau=\epsilon\tau_{xx}+\frac{\epsilon}{2}\tau_{yy}+b_1\tau_x+b_2\tau_y.$$

The recorded parameters are $A=5$, $\nu=0.1$, and $\epsilon=1.2$.


### Five methods compared

All five methods use the same base network and differ in how they impose the boundary condition and define the losses. The standard method penalizes sampled boundary values; the variational method uses a weak residual and Nitsche boundary term; the other three impose zero values through different powers of the ellipse distance factor.

| Method | Trial function | Active loss components |
|---|---|---|
| Standard | $N_\theta$ | strong PDE, boundary, Monte Carlo data |
| Boundary-adapted | $dN_\theta$ | strong PDE, Monte Carlo data |
| Over-enforced | $d^{3/2}N_\theta$ | strong PDE, Monte Carlo data |
| Under-enforced | $d^{1/2}N_\theta$ | strong PDE, Monte Carlo data |
| Variational/Nitsche | $N_\theta$ | weak residual, Nitsche boundary, Monte Carlo data |

The strong residual is $\mathcal Lu_\theta+1$. The standard PINN enforces $u=0$ with sampled boundary values; the three powers of the ellipse distance factor enforce it in the trial function. The Monte Carlo loss compares $u_\theta(z_j)$ with mean exit-time labels $\widehat\tau_j$.

If $J_i$ are the active components in the table, the original adaptive objective is

$$J_{\rm train}=\sum_i e^{-2s_i}J_i+\sum_i s_i,$$

with trainable log scales $s_i$. The checkpoint with the smallest total training objective is retained.


## 2. Choose the run and set problem parameters

Set the run destination and mode here. Problem, sampling, network, training, and evaluation choices are defined with the code that uses them below.

### Runtime setup

In [ ]:
import importlib.util
import json
import math
import random
import secrets
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if importlib.util.find_spec("skfem") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-fem>=11,<13"])

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
import torch
from torch import nn
from skfem import Basis, BilinearForm, ElementTriP1, LinearForm, MeshTri, asm, condense, solve

# Runtime settings are separate from the experiment parameters below.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"  # Change to "float32" if needed.
if DTYPE_NAME not in ("float32", "float64"):
    raise ValueError("DTYPE_NAME must be 'float32' or 'float64'.")
torch.set_default_dtype(torch.float64 if DTYPE_NAME == "float64" else torch.float32)
print("Training device:", DEVICE)

In [ ]:
# Run and output
RUN_MODE = "new"                 # "new" trains a fresh run; "resume" or "evaluate" selects a saved run.
RUN_LABEL = "double_well"
SAVE_TO_DRIVE = False            # True mounts Google Drive when the run folder is prepared; False uses ./runs/double_well.
RUN_REFERENCE = True             # Build the finite-element reference solution.
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_LBFGS = False
DOWNLOAD_RUN_ARCHIVE = True  # Download the ZIP automatically when using local Colab storage.

### Double-well process and exit domain

These physical parameters define the drift, ellipse, and absorbing boundary in the PDE above.

In [ ]:
potential_height = 5.0
well_width = 0.1
diffusion_epsilon = 1.2
well_1_center = (1.0, 0.5)
well_2_center = (-1.0, -0.5)
ellipse_a = 2.0                    # Horizontal semi-axis.
ellipse_b = 1.5                    # Vertical semi-axis.

## 3. Define the network and PDE

Edit `BaseNetwork` in the next code cell using ordinary PyTorch layers and initialization. Keep one scalar output per point. `DoubleWellModel` applies the selected trial function and trainable loss scales; the following cells define the problem-specific drift and PDE residual.


In [ ]:
def smooth_distance(points):
    x, y = points[:, 0], points[:, 1]
    omega = 1.0 - (x / ellipse_a).square() - (y / ellipse_b).square()
    grad_squared = (2 * x / ellipse_a**2).square() + (2 * y / ellipse_b**2).square()
    return omega / torch.sqrt(omega.square() + grad_squared)


# Edit the layers and initialization directly using standard PyTorch code.
# Keep one scalar output per input point.
class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.layers.apply(self.initialize)

    @staticmethod
    def initialize(layer):
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, points):
        return self.layers(points)



class DoubleWellModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        if method == "adapted":
            self.power = 1.0
        elif method == "over":
            self.power = 1.5
        elif method == "under":
            self.power = 0.5
        else:
            self.power = None
        self.network = BaseNetwork()
        if method == "standard":
            loss_names = ("pde", "data", "bc")
        elif method == "variational":
            loss_names = ("var", "data", "bc")
        else:
            loss_names = ("pde", "data")
        initial_scales = {
            name: torch.tensor(adaptive_log_scale_init)
            for name in loss_names
        }
        if method == "variational":
            initial_scales["bc"] = torch.tensor(nitsche_bc_log_scale_init)
        self.log_scales = nn.ParameterDict({
            name: nn.Parameter(value) for name, value in initial_scales.items()
        })

    def forward(self, points):
        raw = self.network(points)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (points.shape[0], 1):
            raise ValueError("BaseNetwork must return one scalar per point: [n] or [n, 1].")
        if self.power is None:
            return raw
        return raw * smooth_distance(points).clamp_min(0).pow(self.power).unsqueeze(1)


### PDE residual

The residual is $\mathcal L\tau_\theta+1$. Its drift and diffusion coefficients are written out here so they can be changed together with the SDE above.

In [ ]:
def strong_residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    drift = drift_torch(x)
    generator = diffusion_epsilon * tau_xx + 0.5 * diffusion_epsilon * tau_yy + (drift * grad).sum(dim=1)
    return generator + 1.0


## 4. Prepare samples and reference data

This section defines the domain samplers and numerical reference. Training and evaluation point counts are set in Section 5; Monte Carlo anchor and path settings follow those choices in Section 6.

### Sample fixed points

Interior points are uniform by area: sample an angle uniformly and a radius as $\sqrt{U}$, then scale by the ellipse semi-axes. Boundary points use a uniform angle; boundary integrals multiply by the ellipse arclength factor. The training, evaluation, and optional L-BFGS sets are each sampled once and saved. Monte Carlo labels use independent Euler–Maruyama paths from uniformly sampled interior anchors.

In [ ]:
def sample_ellipse(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radius = torch.sqrt(torch.rand(count, device=device))
    return torch.stack((ellipse_a * radius * torch.cos(angle), ellipse_b * radius * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((ellipse_a * torch.cos(angle), ellipse_b * torch.sin(angle)), dim=1)


def drift_torch(points):
    x, y = points[..., 0], points[..., 1]
    d1 = (x - well_1_center[0]).square() + (y - well_1_center[1]).square()
    d2 = (x - well_2_center[0]).square() + (y - well_2_center[1]).square()
    e1, e2 = torch.exp(-d1 / well_width**2), torch.exp(-d2 / well_width**2)
    f1, f2 = 1.0 - e1, 1.0 - e2
    dphi_x = potential_height * (
        2 * (x - well_1_center[0]) / well_width**2 * e1 * f2
        + f1 * 2 * (x - well_2_center[0]) / well_width**2 * e2
    )
    dphi_y = potential_height * (
        2 * (y - well_1_center[1]) / well_width**2 * e1 * f2
        + f1 * 2 * (y - well_2_center[1]) / well_width**2 * e2
    )
    return torch.stack((-dphi_x, -dphi_y), dim=-1)

### Finite-element reference resolution

`fem_refinements` controls the triangular mesh resolution for the numerical reference. Higher values take longer.

In [ ]:
fem_refinements = 4


### Drift and FEM reference

The weak reference problem is

$$\int_\Omega\left(\epsilon u_xv_x+\frac{\epsilon}{2}u_yv_y-(b\cdot\nabla u)v\right)dx
=\int_\Omega v\,dx,$$

with homogeneous Dirichlet data. The paper runs used FEniCS on a generated ellipse mesh. The cell below uses continuous piecewise-linear elements in `scikit-fem`, making the Colab setup straightforward while keeping the equation and weak form fixed.


In [ ]:
def drift_numpy(x, y):
    d1 = (x - well_1_center[0])**2 + (y - well_1_center[1])**2
    d2 = (x - well_2_center[0])**2 + (y - well_2_center[1])**2
    e1, e2 = np.exp(-d1 / well_width**2), np.exp(-d2 / well_width**2)
    f1, f2 = 1.0 - e1, 1.0 - e2
    dphi_x = potential_height * (
        2 * (x - well_1_center[0]) / well_width**2 * e1 * f2
        + f1 * 2 * (x - well_2_center[0]) / well_width**2 * e2
    )
    dphi_y = potential_height * (
        2 * (y - well_1_center[1]) / well_width**2 * e1 * f2
        + f1 * 2 * (y - well_2_center[1]) / well_width**2 * e2
    )
    return -dphi_x, -dphi_y


def solve_fem_reference():
    mesh = MeshTri.init_circle(nrefs=fem_refinements).scaled((ellipse_a, ellipse_b))
    basis = Basis(mesh, ElementTriP1())

    @BilinearForm
    def operator(u, v, w):
        bx, by = drift_numpy(w.x[0], w.x[1])
        return (diffusion_epsilon * u.grad[0] * v.grad[0]
                + 0.5 * diffusion_epsilon * u.grad[1] * v.grad[1]
                - (bx * u.grad[0] + by * u.grad[1]) * v)

    @LinearForm
    def forcing(v, w):
        return v

    matrix = asm(operator, basis)
    rhs = asm(forcing, basis)
    coefficients = solve(*condense(matrix, rhs, D=basis.get_dofs()))
    return mesh, basis, coefficients


fem_mesh = fem_basis = fem_coefficients = None
if RUN_REFERENCE:
    fem_mesh, fem_basis, fem_coefficients = solve_fem_reference()
    print(f"FEM reference: {fem_basis.N} P1 degrees of freedom")


## 5. Define losses and experiment settings

Edit the total training point counts, method losses, optimizer, and learning-rate controls here. Section 6 sets the Monte Carlo anchor fraction and path-simulation controls, then creates or loads all labels and fixed points before defining the Adam and optional L-BFGS routines. The final cell in Section 6 starts or loads the selected methods.


### Total points in the training and evaluation sets

These are fixed sets, not mini-batch sizes. Every Adam update uses the full saved training sets, reused by all selected methods. Evaluation and optional L-BFGS use separate saved points.

In [ ]:
training_collocation_points = 384  # Total interior points used in the PDE loss each epoch.
training_boundary_points = 96      # Total boundary points used in the boundary loss each epoch.
evaluation_interior_points = 384   # Separate fixed points used only for progress reporting.
evaluation_boundary_points = 96

### Variational/Nitsche settings

These values affect only the variational/Nitsche method. The test count and length set its Gaussian integration tests; the log scales initialize its learned loss weights. The formula and loss code below show where they enter.

In [ ]:
# Controls used only by the variational/Nitsche method.
n_test_functions = 50  # number of Gaussian test functions
test_length = math.pi * ellipse_a * ellipse_b / 10.0  # width of the Gaussian test functions
adaptive_log_scale_init = -0.5  # initial adaptive PDE/data loss scale
nitsche_bc_log_scale_init = -0.5  # initial Nitsche boundary-loss scale

### Variational/Nitsche loss

With $D=\operatorname{diag}(\epsilon,\epsilon/2)$, integration by parts gives

$$
\int_\Omega \nabla u\cdot D\nabla\phi\,dx
-\int_\Omega(b\cdot\nabla u)\phi\,dx
-\int_{\partial\Omega}(D\nabla u\cdot n)\phi\,ds
=\int_\Omega\phi\,dx.
$$

The symmetric Nitsche term is $-\int_{\partial\Omega}u(D\nabla\phi\cdot n)\,ds$, and $\int_{\partial\Omega}u\phi\,ds$ supplies the separately weighted boundary component. Fifty Gaussian tests are used with length scale $|\Omega|/10$. Volume integrals are estimated by uniform-area sample averages. Boundary angles are sampled uniformly and multiplied by $|x'(\theta)|$, so the boundary terms approximate arclength integrals rather than angular averages.


In [ ]:
test_centers = sample_ellipse(n_test_functions)
def boundary_arclength_factor(points):
    """Return |d(a cos(theta), b sin(theta))/d theta| at boundary points."""
    x, y = points[:, 0], points[:, 1]
    return torch.sqrt((ellipse_a * y / ellipse_b).square() + (ellipse_b * x / ellipse_a).square())


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]
    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * ellipse_a * ellipse_b
    drift = drift_torch(interior.detach())
    diffusion_term = (diffusion_epsilon * grad_u[:, None, 0] * grad_phi[:, :, 0]
                      + 0.5 * diffusion_epsilon * grad_u[:, None, 1] * grad_phi[:, :, 1])
    drift_term = -(drift * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion_term + drift_term).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal_raw = torch.stack((boundary[:, 0] / ellipse_a**2, boundary[:, 1] / ellipse_b**2), dim=1)
    normal = normal_raw / torch.linalg.norm(normal_raw, dim=1, keepdim=True)
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    flux_u = (
        diffusion_epsilon * grad_u_b[:, 0] * normal[:, 0]
        + 0.5 * diffusion_epsilon * grad_u_b[:, 1] * normal[:, 1]
    )
    ds_dtheta = boundary_arclength_factor(boundary)
    consistency = -2 * math.pi * (flux_u[:, None] * phi_b * ds_dtheta[:, None]).mean(dim=0)
    flux_phi = (diffusion_epsilon * grad_phi_b[:, :, 0] * normal[:, None, 0]
                + 0.5 * diffusion_epsilon * grad_phi_b[:, :, 1] * normal[:, None, 1])

    symmetry = -2 * math.pi * (u_b[:, None] * flux_phi * ds_dtheta[:, None]).mean(dim=0)
    raw_penalty = 2 * math.pi * (u_b[:, None] * phi_b * ds_dtheta[:, None]).mean(dim=0)
    return weak + consistency + symmetry, raw_penalty


def adaptive_objective(model, components):
    weighted_components = [
        torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name]
        for name, value in components.items()
    ]
    return sum(weighted_components)


def component_losses(model, method, interior, boundary, data_x, data_y):
    data_loss = (model(data_x).squeeze(1) - data_y).square().mean()
    if method == "variational":
        weak, penalty = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data_loss, "bc": penalty.square().mean()}
    losses = {"pde": strong_residual(model, interior).square().mean(), "data": data_loss}
    if method == "standard":
        values = model(boundary).squeeze(1).square()
        weights = boundary_arclength_factor(boundary)
        losses["bc"] = (values * weights).sum() / weights.sum()
    return losses


### Optional L-BFGS settings

L-BFGS is a second optimization stage that starts from the best Adam checkpoint. It runs only when `RUN_LBFGS = True`; otherwise the best Adam checkpoint is the result. The settings below control its fixed points and stopping limits.

In [ ]:
# Read only by the optional L-BFGS refinement below.
lbfgs_interior_points = 384  # fixed interior points for refinement
lbfgs_boundary_points = 96  # fixed boundary points for refinement
lbfgs_iterations = 20  # maximum optimizer iterations
lbfgs_lr = 1.0  # L-BFGS step-size parameter
lbfgs_history_size = 500  # curvature pairs retained
lbfgs_tolerance_grad = 0.0  # gradient stopping tolerance
lbfgs_tolerance_change = 0.0  # loss-change stopping tolerance

### Adam optimizer and learning-rate schedule

These settings are used by `train_method`, defined in Section 6 after the data are prepared. Each epoch makes one full-batch Adam update on the fixed points, then advances the exponential schedule. Change the values here to alter training; report and checkpoint intervals control progress output and restart files.

In [ ]:
# The Adam loop in Section 6 reads these settings.
learning_rate = 5e-4  # initial Adam step size
adam_beta_1 = 0.9  # exponential average of gradients
adam_beta_2 = 0.999  # exponential average of squared gradients
adam_epsilon = 1e-8  # stabilizer in Adam's denominator
weight_decay = 0.0  # parameter penalty; zero disables it
adam_epochs = 30  # maximum full-data updates
final_learning_rate_factor = 1e-3  # final step size as a fraction of the initial rate
report_fraction = 0.1  # Fraction of planned epochs between progress reports.
if adam_epochs < 1:
    raise ValueError("adam_epochs must be at least 1.")
if not 0 < report_fraction <= 1:
    raise ValueError("report_fraction must be in (0, 1].")
report_every = max(1, math.ceil(report_fraction * adam_epochs))  # Derive interval from total run length.
if checkpoint_every < 1:
    raise ValueError("checkpoint_every must be at least 1.")
checkpoint_every = 10  # restart-save interval in epochs

## 6. Prepare data, then train

First define the path simulator and create or load the Monte Carlo labels and fixed point sets. Only after those data are ready does the notebook define the optimizer loops. The final training cell starts or loads the selected methods.


### Monte Carlo labels for mean exit time

At each anchor $z$, the notebook simulates $M$ independent Euler–Maruyama paths for the SDE in Section 1. If $T_j$ is a path's first discrete exit time and $T_{\max}$ is the simulation cap, the label is

$$
\widehat\tau_M(z)=\frac1M\sum_{j=1}^M\min(T_j,T_{\max}).
$$

This estimates $\tau(z)=\mathbb E_z[T]$ by the Feynman–Kac representation. The PDE loss uses the fixed collocation points; the data loss compares the network with these labels at the anchors, and both enter the objective before an optimizer update. A path still inside at the cap is recorded as censored. Replacing its unknown exit time by the cap tends to lower the estimated mean when censoring is substantial, so the saved censored fraction shows when a longer cap may be needed. For the role of noisy Feynman–Kac labels in PINN training, see [Tepakbong et al., “Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision” (ICML 2026)](https://proceedings.mlr.press/v306/tepakbong26a.html).

In [ ]:
# Fraction of PDE collocation points used as Monte Carlo starting locations.
data_anchor_fraction = 0.05
# Point counts are integers: int takes the whole-point part of the product.
data_anchors = int(data_anchor_fraction * training_collocation_points)
if data_anchors < 1:
    raise ValueError("Increase data_anchor_fraction so at least one anchor is used.")
# More independent paths per anchor reduce Monte Carlo noise and take longer.
paths_per_anchor = 20
mc_dt = 2e-3
mc_max_time = 2.0

### Simulate the paths

The following function applies one Euler–Maruyama step to every live path, records the first discrete exit time, and stops at the cap. It returns the capped mean exit-time estimate and the fraction of paths that were still inside. The data-preparation cell below calls it for every saved anchor before training starts.

In [ ]:
# Simulate independent Euler–Maruyama paths from each anchor.
# Paths still alive at the cap contribute the cap to the mean; their share is saved as censored.
@torch.no_grad()
def monte_carlo_exit_time(x0, n_paths, dt, max_time):
    x0 = x0.to(DEVICE)
    batch = x0.shape[0]
    positions = x0[:, None, :].expand(batch, n_paths, 2).clone()
    elapsed = torch.zeros(batch, n_paths, device=DEVICE)
    alive = torch.ones(batch, n_paths, dtype=torch.bool, device=DEVICE)
    max_steps = int(max_time / dt)
    interval = max(1, max_steps // 100)
    noise_scale = torch.tensor(
        [math.sqrt(2 * diffusion_epsilon * dt), math.sqrt(diffusion_epsilon * dt)],
        device=DEVICE,
    )
    started = time.time()
    for step in range(max_steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current + drift_torch(current) * dt + torch.randn_like(current) * noise_scale
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        active = positions[alive]
        exited = (active[:, 0] / ellipse_a).square() + (active[:, 1] / ellipse_b).square() >= 1.0
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == max_steps:
            print(
                f"MC step {step + 1:6d}/{max_steps} | "
                f"alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s",
                flush=True,
            )
    if alive.any():
        print(f"Censored at {max_time:g}: {int(alive.sum())}/{alive.numel()} paths")
    return elapsed.mean(dim=1), alive.float().mean(dim=1)

### Select the run folder and seeds

This cell applies the `RUN_MODE` and `SAVE_TO_DRIVE` choices from Section 2. A new run gets its own folder and fresh seeds; resume and evaluation reuse a saved run. Drive connects only when selected. This step chooses storage and seeds; it does not generate labels or train a network.


In [ ]:
# "new" creates a fresh run; "resume" continues it; "evaluate" loads results without training.
if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'.")

# Mount Drive only when requested; otherwise use the local runs/ directory.
if SAVE_TO_DRIVE:
    if drive is None:
        raise RuntimeError("Google Drive saving is available in Colab; set SAVE_TO_DRIVE = False for local runs.")
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    output_base = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/double_well")
else:
    output_base = Path("runs/double_well")
output_base.mkdir(parents=True, exist_ok=True)

# A new run gets fresh seeds and a unique timestamped folder.
# New runs create a unique folder and draw fresh training and Monte Carlo seeds.
if RUN_MODE == "new":
    # These seeds are generated once for this new run and saved in its manifest.
    SEED = secrets.randbits(32)
    MC_SEED = secrets.randbits(32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    run_dir = output_base / f"{RUN_LABEL}_{run_stamp}_seed{SEED}"
    run_dir.mkdir(parents=False, exist_ok=False)
    saved_manifest = None
else:
    # Resume/evaluate select an existing saved run folder.
    candidates = sorted(
        (path for path in output_base.glob(f"{RUN_LABEL}_*") if path.is_dir() and (path / "run_manifest.json").is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No saved run for {RUN_LABEL!r} was found in {output_base}.")
    if len(candidates) == 1:
        run_dir = candidates[0]
    else:
        print("Choose the saved run:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        choice = int(input(f"Run number [1-{len(candidates)}]: "))
        if not 1 <= choice <= len(candidates):
            raise ValueError("Choose one of the listed run numbers.")
        run_dir = candidates[choice - 1]
    manifest_path = run_dir / "run_manifest.json"
    saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    SEED = int(saved_manifest["seed"])
    MC_SEED = int(saved_manifest.get("mc_seed", saved_manifest.get("monte_carlo_seed")))

RESUME = RUN_MODE == "resume"
LOAD_SAVED_MODELS = RUN_MODE == "evaluate"
print(f"Mode: {RUN_MODE} | output: {run_dir} | training seed: {SEED} | Monte Carlo seed: {MC_SEED}")

def seed_all(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def atomic_csv_save(frame, path):
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng_state(state):
    if not state:
        raise ValueError("Checkpoint has no random-generator state; exact resume is unavailable.")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])

seed_all(SEED)


### Record the choices for this run

This cell copies the problem, sample counts, methods, optimizer, seeds, and reporting choices into the run's JSON record. Resume and evaluation compare the saved choices with the current ones before reusing data or checkpoints.


In [ ]:
# Save exact settings so a resumed run cannot mix data from different choices.
experiment_settings = {
    "run_label": RUN_LABEL, "seed": SEED, "mc_seed": MC_SEED,
    "potential_height": potential_height, "well_width": well_width, "diffusion_epsilon": diffusion_epsilon,
    "well_1_center": list(well_1_center), "well_2_center": list(well_2_center),
    "ellipse_a": ellipse_a, "ellipse_b": ellipse_b, "fem_refinements": fem_refinements,
    "training_collocation_points": training_collocation_points, "training_boundary_points": training_boundary_points,
    "evaluation_interior_points": evaluation_interior_points, "evaluation_boundary_points": evaluation_boundary_points,
    "lbfgs_interior_points": lbfgs_interior_points, "lbfgs_boundary_points": lbfgs_boundary_points,
    "data_anchors": data_anchors, "data_anchor_fraction": data_anchor_fraction, "paths_per_anchor": paths_per_anchor, "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "adam_epochs": adam_epochs, "report_fraction": report_fraction,
    "report_every": report_every, "checkpoint_every": checkpoint_every, "learning_rate": learning_rate,
    "adam_beta_1": adam_beta_1, "adam_beta_2": adam_beta_2, "adam_epsilon": adam_epsilon, "weight_decay": weight_decay,
    "final_learning_rate_factor": final_learning_rate_factor, "methods": list(METHODS),
    "run_reference": RUN_REFERENCE, "run_lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations,
    "lbfgs_lr": lbfgs_lr, "lbfgs_history_size": lbfgs_history_size,
    "n_test_functions": n_test_functions, "test_length": test_length,
    "device": str(DEVICE), "dtype": DTYPE_NAME,

}
experiment_settings = json.loads(json.dumps(experiment_settings))


### Save and restore run files

These small helpers update the run record and write arrays, figures, checkpoints, and histories safely. The preparation, training, and evaluation cells call them; they do not start an experiment.


In [ ]:
# Check saved choices before resuming, and update the run record as work completes.
def require_matching_settings(record, expected, saved_path):
    """Stop before reusing data or checkpoints made with different settings."""
    saved = record.get("settings", record)
    if any(saved.get(key) != value for key, value in expected.items()):
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Restore those settings or start a new run."
        )

manifest_path = run_dir / "run_manifest.json"
if RUN_MODE == "new":
    write_run_manifest_record = {
        "schema_version": 2,
        "run_id": run_dir.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "status": "created",
        "artifacts": [],
    }
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(write_run_manifest_record, stream, indent=2)
else:
    write_run_manifest_record = json.loads(manifest_path.read_text(encoding="utf-8"))
    require_matching_settings(write_run_manifest_record, experiment_settings, manifest_path)

# Refresh the run record with its status and saved files.
def write_run_manifest(status):
    """Update the run record after a stage finishes."""
    write_run_manifest_record["status"] = status
    write_run_manifest_record["updated_utc"] = datetime.now(timezone.utc).isoformat()
    write_run_manifest_record["artifacts"] = sorted(
        str(path.relative_to(run_dir))
        for path in run_dir.rglob("*")
        if path.is_file() and path != manifest_path and ".tmp" not in path.name
    )
    temporary_path = manifest_path.with_name(manifest_path.name + ".tmp")
    temporary_path.write_text(json.dumps(write_run_manifest_record, indent=2), encoding="utf-8")
    temporary_path.replace(manifest_path)

def atomic_npz_save(path, **arrays):
    """Save arrays through a temporary file so an interruption leaves no partial result."""
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary_path.replace(path)

def atomic_figure_save(figure, path, **kwargs):
    """Save a complete figure before replacing the previous file."""
    temporary_path = path.with_name(path.stem + ".tmp" + path.suffix)
    figure.savefig(temporary_path, **kwargs)
    temporary_path.replace(path)

def require_artifact_files(paths):
    missing = [path for path in paths if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError("Missing or empty run outputs: " + ", ".join(map(str, missing)))

write_run_manifest("preparing")
print(f"Run folder: {run_dir} | seed: {SEED} | Monte Carlo seed: {MC_SEED}")


### Prepare or load the shared data

This cell loads the selected run's saved arrays or creates them for a new run. A new run simulates exit times at the Monte Carlo anchors, records the fraction of paths reaching the cap, and samples the fixed PDE, boundary, evaluation, and optional L-BFGS points. They are saved together so training, restart, and later plots use the same data. No network is trained here; the following cell starts or resumes the selected methods.

In [ ]:
# Load or create the frozen arrays for this run.
data_path = run_dir / "shared_data.pt"
data_settings = {
    "format_version": 5,
    "seed": SEED, "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED,
    "potential_height": potential_height, "well_width": well_width,
    "diffusion_epsilon": diffusion_epsilon, "ellipse_a": ellipse_a, "ellipse_b": ellipse_b,
    "well_1_center": well_1_center, "well_2_center": well_2_center,
    "fem_refinements": fem_refinements,
    "data_anchors": data_anchors, "data_anchor_fraction": data_anchor_fraction, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "training_collocation_points": training_collocation_points,
    "training_boundary_points": training_boundary_points,
    "evaluation_interior_points": evaluation_interior_points,
    "evaluation_boundary_points": evaluation_boundary_points,
    "lbfgs_interior_points": lbfgs_interior_points,
    "lbfgs_boundary_points": lbfgs_boundary_points,
    "n_test_functions": n_test_functions,
}

write_run_manifest("preparing")

# Resume/evaluate load this run's saved arrays and verify the settings before use.
# Resume/evaluation reuse the saved labels; only a new run samples them.
if RUN_MODE in {"resume", "evaluate"}:
    if not data_path.is_file():
        raise FileNotFoundError(f"Saved run data is missing: {data_path}")
    shared = torch.load(data_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(shared, data_settings, data_path)
    x_data = shared["x"].to(DEVICE)
    tau_data = shared["tau"].to(DEVICE)
    data_censored = shared["censored"].to(DEVICE)
    training_interior = shared["training_interior"].to(DEVICE)
    training_boundary = shared["training_boundary"].to(DEVICE)
    evaluation_interior = shared["evaluation_interior"].to(DEVICE)
    evaluation_boundary = shared["evaluation_boundary"].to(DEVICE)
    lbfgs_interior = shared["lbfgs_interior"].to(DEVICE)
    lbfgs_boundary = shared["lbfgs_boundary"].to(DEVICE)
    test_centers = shared["test_centers"].to(DEVICE)
    print("Loaded this run's labels and fixed point sets:", data_path)
else:
    # Do not overwrite an existing run's saved arrays.
    if data_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing run data: {data_path}")
    seed_all(MC_SEED)
    # Sample anchors and estimate their mean exit times.
# Simulate the Feynman–Kac labels before any network is trained.
    x_data = sample_ellipse(data_anchors)
    tau_data, data_censored = monte_carlo_exit_time(
        x_data, paths_per_anchor, mc_dt, mc_max_time
    )
    # Sample the fixed PDE, boundary, evaluation, and refinement sets once.
    training_interior = sample_ellipse(training_collocation_points)
    training_boundary = sample_boundary(training_boundary_points)
    evaluation_interior = sample_ellipse(evaluation_interior_points)
    evaluation_boundary = sample_boundary(evaluation_boundary_points)
    lbfgs_interior = sample_ellipse(lbfgs_interior_points)
    lbfgs_boundary = sample_boundary(lbfgs_boundary_points)
    test_centers = sample_ellipse(n_test_functions)
    # Store CPU copies together so later training and plots reuse identical arrays.
# Save labels, censoring fractions, and every point set used later.
    shared = {
        "x": x_data.cpu(), "tau": tau_data.cpu(), "censored": data_censored.cpu(),
        "training_interior": training_interior.cpu(),
        "training_boundary": training_boundary.cpu(),
        "evaluation_interior": evaluation_interior.cpu(),
        "evaluation_boundary": evaluation_boundary.cpu(),
        "lbfgs_interior": lbfgs_interior.cpu(),
        "lbfgs_boundary": lbfgs_boundary.cpu(),
        "test_centers": test_centers.cpu(),
        "settings": data_settings,
    }
    atomic_torch_save(shared, data_path)
    print("Saved this run's labels and fixed point sets:", data_path)

seed_all(SEED)
print(f"Shared labels: {len(x_data)} anchors × {paths_per_anchor} paths")
atomic_npz_save(
    run_dir / "shared_data.npz",
    anchors=x_data.cpu().numpy(),
    mean_exit_times=tau_data.cpu().numpy(),
    censored_fraction=data_censored.cpu().numpy(),
    training_interior=training_interior.cpu().numpy(),
    training_boundary=training_boundary.cpu().numpy(),
    evaluation_interior=evaluation_interior.cpu().numpy(),
    evaluation_boundary=evaluation_boundary.cpu().numpy(),
    lbfgs_interior=lbfgs_interior.cpu().numpy(),
    lbfgs_boundary=lbfgs_boundary.cpu().numpy(),
    test_centers=test_centers.cpu().numpy(),
    training_seed=np.asarray(SEED, dtype=np.uint64),
    monte_carlo_seed=np.asarray(MC_SEED, dtype=np.uint64),
)
write_run_manifest("prepared")
print(f"Censored fraction: mean={float(data_censored.mean()):.2%}, max={float(data_censored.max()):.2%}")

### Adam update loop

This cell defines `train_method(method)`; it does not start training by itself. When the final cell calls it, each epoch computes the selected method's PDE, boundary, and Monte Carlo losses on the saved points, backpropagates their combined objective, and takes one Adam step. It then evaluates on fixed points and writes the best checkpoint, restart state, and epoch history at the configured intervals.

In [ ]:
def detached_row(components):
    return {name: float(value.detach()) for name, value in components.items()}


def train_method(method):
    """Train or resume one method using the shared labels and point sets."""
    # Each method has its own model and restart files, but shares the saved data.
    method_dir = run_dir / method
    if RUN_MODE == "new" and method_dir.exists():
        raise FileExistsError(f"Refusing to reuse existing method folder: {method_dir}")
    method_dir.mkdir(parents=True, exist_ok=(RUN_MODE == "resume"))
    model = DoubleWellModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(adam_beta_1, adam_beta_2),
        eps=adam_epsilon,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer,
        gamma=final_learning_rate_factor ** (1.0 / adam_epochs),
    )
    history, start_epoch, best_score = [], 0, math.inf
    last_path, best_path = method_dir / "last.pt", method_dir / "best_adam.pt"
    final_path = method_dir / "final.pt"
    checkpoint_settings = {**experiment_settings, "method": method}
    lbfgs_history = []
    if RUN_MODE == "resume" and final_path.exists():
        raise ValueError(f"{final_path} is already complete; set RUN_MODE='evaluate' to reuse it.")
    if RUN_MODE == "resume" and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, checkpoint_settings, last_path)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        restore_rng_state(state.get("rng_state"))
        print(f"{method}: resuming after epoch {start_epoch}")
    elif RUN_MODE == "resume" and best_path.exists():
        raise ValueError(f"{best_path} exists without a resumable last.pt checkpoint; refusing an inexact resume.")

    # The main Adam loop below performs one full-data update per epoch.
    started = time.time()
    # Main Adam loop: one full-batch update per epoch on the saved training sets.
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        # Compute the selected method's loss terms, then update its network and scales.
        model.train()
        optimizer.zero_grad()
        interior = training_interior
        boundary = training_boundary
        components = component_losses(model, method, interior, boundary, x_data, tau_data)
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()
        # Compare epochs and methods on the same held-out points.
        model.eval()
        validation = component_losses(model, method, evaluation_interior, evaluation_boundary, x_data, tau_data)
        score = float(adaptive_objective(model, validation).detach())
        row = {
            "epoch": epoch,
            "objective": float(objective.detach()),
            "checkpoint_objective": score,
            **detached_row(components),
        }
        history.append(row)
        # Keep the model with the lowest score on the fixed evaluation points.
        if score < best_score:
            best_score = score
            best_checkpoint = {
                "model": model.state_dict(),
                "epoch": epoch,
                "score": score,
                "settings": checkpoint_settings,
            }
            atomic_torch_save(best_checkpoint, best_path)
        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{key} {value:.3e}" for key, value in detached_row(components).items())
            print(f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | {time.time() - started:.1f}s", flush=True)
        # Periodic restart files let training continue after an interruption.
        if epoch % checkpoint_every == 0 or epoch == adam_epochs:
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "history": history,
                "epoch": epoch,
                "best_score": best_score,
                "settings": checkpoint_settings,
                "rng_state": capture_rng_state(),
            }
            atomic_torch_save(checkpoint, last_path)
            atomic_csv_save(pd.DataFrame(history), method_dir / "training_history.csv")
            write_run_manifest("training")

    # Optional L-BFGS refines the best Adam checkpoint when explicitly enabled.
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["model"])
    lbfgs_history = run_lbfgs_stage(model, method, method_dir, x_data, tau_data, lbfgs_history)
    atomic_torch_save({
        "model": model.state_dict(), "history": history, "lbfgs_history": lbfgs_history,
        "method": method, "run_label": RUN_LABEL,
        "settings": {**checkpoint_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations},
        "rng_state": capture_rng_state(),
    }, final_path)
    history_frame = pd.DataFrame(history)
    atomic_csv_save(history_frame, method_dir / "training_history.csv")
    return model, history_frame


### Optional L-BFGS refinement

This helper is called only after the Adam loop and only when `RUN_LBFGS = True`. It starts from the best Adam checkpoint and uses the L-BFGS settings above. Leave it disabled to finish after Adam.

In [ ]:
def run_lbfgs_stage(model, method, method_dir, x_data, tau_data, lbfgs_history):
    """Optionally refine the best Adam model using the fixed L-BFGS points."""
    if RUN_LBFGS:
        fixed_interior, fixed_boundary = lbfgs_interior, lbfgs_boundary
        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=lbfgs_lr,
            max_iter=lbfgs_iterations,
            history_size=lbfgs_history_size,
            tolerance_grad=lbfgs_tolerance_grad,
            tolerance_change=lbfgs_tolerance_change,
            line_search_fn="strong_wolfe",
        )
        calls = 0
        # L-BFGS may evaluate this closure repeatedly while selecting an update.
        def closure():
            nonlocal calls
            calls += 1
            lbfgs.zero_grad()
            parts = component_losses(model, method, fixed_interior, fixed_boundary, x_data, tau_data)
            loss = adaptive_objective(model, parts)
            loss.backward()
            lbfgs_history.append({"call": calls, "objective": float(loss.detach()), **detached_row(parts)})
            if calls % 100 == 0:
                atomic_csv_save(pd.DataFrame(lbfgs_history), method_dir / "lbfgs_history.csv")
                write_run_manifest("training")
                print(method, "L-BFGS call", calls, detached_row(parts), flush=True)
            return loss

        lbfgs.step(closure)
        atomic_csv_save(pd.DataFrame(lbfgs_history), method_dir / "lbfgs_history.csv")
        write_run_manifest("training")
    return lbfgs_history

### Start or load the selected methods

This is the cell that actually starts training. In `new` or `resume` mode it calls `train_method` for each selected method, using the Monte Carlo labels and fixed points prepared above. In `evaluate` mode it loads completed checkpoints and does not train.

In [ ]:
trained = {}
histories = {}
lbfgs_histories = {}
if RUN_MODE in {"new", "resume"}:
    write_run_manifest("training")
    if RUN_MODE == "evaluate":
        raise ValueError("Set RUN_MODE to 'new' or 'resume' before training.")
    for method in METHODS:
        trained[method], histories[method] = train_method(method)
    print("Training complete:", ", ".join(trained))
    write_run_manifest("trained")
elif RUN_MODE == "evaluate":
    write_run_manifest("evaluating")
    for method in METHODS:
        final_path = run_dir / method / "final.pt"
        if not final_path.is_file():
            raise FileNotFoundError(f"Completed model is missing: {final_path}")
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        expected = {**experiment_settings, "method": method, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}
        require_matching_settings(state, expected, final_path)
        model = DoubleWellModel(method).to(DEVICE)
        model.load_state_dict(state["model"])
        trained[method] = model
        histories[method] = pd.DataFrame(state["history"])
        lbfgs_histories[method] = pd.DataFrame(state.get("lbfgs_history", []))
        if RUN_LBFGS and not lbfgs_histories[method].empty:
            atomic_csv_save(lbfgs_histories[method], run_dir / method / "lbfgs_history.csv")
    print("Loaded completed models from this run:", ", ".join(trained))

if trained:
    for method, history in histories.items():
        method_dir = run_dir / method
        method_dir.mkdir(parents=True, exist_ok=True)
        atomic_csv_save(history, method_dir / "training_history.csv")

## 7. Evaluate and report

The notebook compares each method with the finite-element reference and reports relative $L^1$, $L^2$, $H^1$, and $H^2$ errors. Evaluation saves the numerical arrays, metrics, and figures alongside the checkpoints, histories, samples, and labels. It then creates a ZIP containing the complete run. With Drive selected, the ZIP stays beside the run; with local Colab storage, `DOWNLOAD_RUN_ARCHIVE` controls automatic download.

### Evaluation grid, plots, and sample trajectory

These settings control only the displayed evaluation grid and optional example path. They do not change training samples, seeds, or model fitting. The evaluation code saves the grid resolution and path settings beside the figures.

In [ ]:
evaluation_grid_n = 200
plot_dpi = 180
trajectory_start = (0.0, 0.0)
trajectory_dt = 1e-3
trajectory_max_time = 20.0
trajectory_seed_offset = 1

In [ ]:
grid_n = evaluation_grid_n
gx, gy = np.meshgrid(np.linspace(-ellipse_a, ellipse_a, grid_n), np.linspace(-ellipse_b, ellipse_b, grid_n))
if fem_mesh is not None:
    triangulation = mtri.Triangulation(fem_mesh.p[0], fem_mesh.p[1], fem_mesh.t.T)
    ellipse_mask = triangulation.get_trifinder()(gx, gy) >= 0
else:
    ellipse_mask = (gx / ellipse_a)**2 + (gy / ellipse_b)**2 < 1.0
grid_points = np.column_stack((gx[ellipse_mask], gy[ellipse_mask]))


def fem_values(points):
    """Evaluate the finite-element reference at arbitrary points."""
    if fem_basis is None:
        raise RuntimeError("Run the finite-element reference cell first.")
    return np.asarray(fem_basis.probes(np.asarray(points).T) @ fem_coefficients)


def evaluate_model(model):
    """Evaluate one model on the common grid and compute relative errors."""
    model.eval()
    with torch.no_grad():
        prediction = model(torch.as_tensor(grid_points, device=DEVICE)).squeeze(1).cpu().numpy()
    truth = fem_values(grid_points)
    predicted_grid, reference_grid = np.zeros_like(gx), np.zeros_like(gx)
    predicted_grid[ellipse_mask], reference_grid[ellipse_mask] = prediction, truth
    dx, dy = 2 * ellipse_a / (grid_n - 1), 2 * ellipse_b / (grid_n - 1)
    error = predicted_grid - reference_grid
    error_y, error_x = np.gradient(error, dy, dx)
    error_xx = np.gradient(error_x, dx, axis=1)
    error_yy = np.gradient(error_y, dy, axis=0)
    error_xy = np.gradient(error_y, dx, axis=1)
    truth_y, truth_x = np.gradient(reference_grid, dy, dx)
    truth_xx = np.gradient(truth_x, dx, axis=1)
    truth_yy = np.gradient(truth_y, dy, axis=0)
    truth_xy = np.gradient(truth_y, dx, axis=1)
    rel_l1 = np.mean(np.abs(prediction - truth)) / np.mean(np.abs(truth))
    rel_l2 = np.sqrt(np.mean((prediction - truth)**2) / np.mean(truth**2))
    h1_error = np.mean(
        error[ellipse_mask]**2
        + error_x[ellipse_mask]**2
        + error_y[ellipse_mask]**2
    )
    h1_reference = np.mean(
        reference_grid[ellipse_mask]**2
        + truth_x[ellipse_mask]**2
        + truth_y[ellipse_mask]**2
    )
    rel_h1 = np.sqrt(h1_error / h1_reference)
    h2_error = np.mean(
        error[ellipse_mask]**2
        + error_x[ellipse_mask]**2
        + error_y[ellipse_mask]**2
        + error_xx[ellipse_mask]**2
        + error_yy[ellipse_mask]**2
        + 2 * error_xy[ellipse_mask]**2
    )
    h2_reference = np.mean(
        reference_grid[ellipse_mask]**2
        + truth_x[ellipse_mask]**2
        + truth_y[ellipse_mask]**2
        + truth_xx[ellipse_mask]**2
        + truth_yy[ellipse_mask]**2
        + 2 * truth_xy[ellipse_mask]**2
    )
    rel_h2 = np.sqrt(h2_error / h2_reference)
    metrics = {
        "L1 relative": rel_l1,
        "L2 relative": rel_l2,
        "H1 relative": rel_h1,
        "H2 relative": rel_h2,
    }
    return predicted_grid, reference_grid, metrics


### Compute errors and save numerical results

The following cell evaluates each method against the same finite-element reference and saved points. It writes the numerical arrays and error metrics before plotting.


In [ ]:
if not trained or fem_basis is None:
    raise RuntimeError("Load trained models and the finite-element reference before evaluation.")

# Compare all methods on the common grid and saved anchor points.
write_run_manifest("evaluating")
evaluation_settings = {
    "grid_size": evaluation_grid_n,
    "plot_dpi": plot_dpi,
    "trajectory_start": list(trajectory_start),
    "trajectory_dt": trajectory_dt,
    "trajectory_max_time": trajectory_max_time,
    "trajectory_seed_offset": trajectory_seed_offset,
}
(run_dir / "evaluation_settings.json").write_text(
    json.dumps(evaluation_settings, indent=2) + "\n", encoding="utf-8"
)
fields, metrics, reference_grid = {}, {}, None
for name, model in trained.items():
    fields[name], reference_grid, metrics[name] = evaluate_model(model)
display(pd.DataFrame(metrics).T)
evaluation_dir = run_dir / "evaluation"
evaluation_dir.mkdir(exist_ok=True)
error_fields = {name: field - reference_grid for name, field in fields.items()}
metric_frame = pd.DataFrame(metrics).T.rename_axis("method").reset_index()
atomic_csv_save(metric_frame, evaluation_dir / "metrics.csv")
atomic_npz_save(
    evaluation_dir / "fields_and_reference.npz",
    gx=gx, gy=gy, grid_points=grid_points,
    ellipse_mask=ellipse_mask.astype(bool), reference=reference_grid,
    **{f"prediction_{name}": field for name, field in fields.items()},
    **{f"signed_error_{name}": error for name, error in error_fields.items()},
    **{f"absolute_error_{name}": np.abs(error) for name, error in error_fields.items()},
)
figures = {"FEM reference": reference_grid, **fields}


### Plot solution fields and common errors

The panels use shared solution and absolute-error color scales. The finite-element reference and each method are shown on the same grid.


In [ ]:
# Draw the reference and method predictions on matching axes.
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
for axis, (name, values) in zip(axes.flat, figures.items()):
    plot = axis.contourf(gx, gy, values, levels=50)
    axis.set_title(name)
    axis.set_aspect("equal")
    fig.colorbar(plot, ax=axis)
for axis in axes.flat[len(figures):]:
    axis.axis("off")
atomic_figure_save(fig, evaluation_dir / "fields.png", dpi=plot_dpi, bbox_inches="tight")
plt.show()


### Verify and archive the complete run

This cell checks that checkpoints, sampled arrays, metrics, and figures are saved, then creates the ZIP archive containing the complete run.


In [ ]:
# Check the saved artifacts before packaging the run.
write_run_manifest("evaluating")
required_paths = [
    data_path, run_dir / "shared_data.npz",
    evaluation_dir / "metrics.csv",
    evaluation_dir / "fields_and_reference.npz",
    evaluation_dir / "fields.png",
]
for method in trained:
    method_dir = run_dir / method
    required_paths.extend([
        method_dir / "best_adam.pt", method_dir / "last.pt",
        method_dir / "final.pt", method_dir / "training_history.csv",
    ])
    if RUN_LBFGS:
        required_paths.append(method_dir / "lbfgs_history.csv")
require_artifact_files(required_paths)
required_keys = {"gx", "gy", "grid_points", "ellipse_mask", "reference"}
for method in trained:
    required_keys.update({f"prediction_{method}", f"signed_error_{method}", f"absolute_error_{method}"})
with np.load(run_dir / "shared_data.npz", allow_pickle=False) as saved_data:
    missing_data = {
        "anchors", "mean_exit_times", "censored_fraction",
        "evaluation_interior", "evaluation_boundary",
        "training_seed", "monte_carlo_seed",
    }.difference(saved_data.files)
if missing_data:
    raise RuntimeError("Shared-data archive is incomplete; missing arrays: " + ", ".join(sorted(missing_data)))
with np.load(evaluation_dir / "fields_and_reference.npz", allow_pickle=False) as saved_arrays:
    missing_keys = required_keys.difference(saved_arrays.files)
if missing_keys:
    raise RuntimeError("Evaluation archive is incomplete; missing arrays: " + ", ".join(sorted(missing_keys)))
write_run_manifest("complete")

import shutil
archive_base = run_dir.parent / f"{run_dir.name}_archive"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_dir)
print("Complete run archive:", archive_path)
if not SAVE_TO_DRIVE and DOWNLOAD_RUN_ARCHIVE:
    try:
        from google.colab import files
    except ImportError:
        print("Automatic download is available in Colab; the ZIP is saved at the path above.")
    else:
        files.download(archive_path)


### Optional exit trajectory

The next code cells set up one explanatory Euler–Maruyama path. It does not affect the saved Monte Carlo labels or training.

In [ ]:
def simulate_double_well_trajectory(
    start=(0.0, 0.0), *, dt=1e-3, max_time=20.0, seed=SEED + 1,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = (point[0] / ellipse_a)**2 + (point[1] / ellipse_b)**2 >= 1.0
    noise = np.array([math.sqrt(2 * diffusion_epsilon * dt), math.sqrt(diffusion_epsilon * dt)])
    for _ in range(int(max_time / dt)):
        if exited:
            break
        bx, by = drift_numpy(point[0], point[1])
        point = point + np.array([bx, by]) * dt + noise * rng.standard_normal(2)
        path.append(point.copy())
        exited = (point[0] / ellipse_a)**2 + (point[1] / ellipse_b)**2 >= 1.0
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_double_well_trajectory(
    start=trajectory_start, dt=trajectory_dt, max_time=trajectory_max_time,
    seed=SEED + trajectory_seed_offset,
)
angle = np.linspace(0.0, 2.0 * np.pi, 300)
fig, axis = plt.subplots(figsize=(6, 5))
axis.plot(trajectory[:, 0], trajectory[:, 1], lw=1.0)
axis.plot(ellipse_a * np.cos(angle), ellipse_b * np.sin(angle), "k--", label="exit boundary")
axis.scatter(*trajectory[0], label="start")
axis.scatter(*trajectory[-1], marker="x", label="exit" if trajectory_exited else "time cap")
axis.set(
    aspect="equal",
    title=f"Double-well path: {'exit' if trajectory_exited else 'censored'} "
          f"at t={trajectory_time:.3g}",
)
axis.legend()
plt.show()


### Historical paper-run summary

These aggregate values were retained as output in the original result notebook. The underlying `dw.txt` log is absent. They used the FEniCS reference and were produced before the corrected checkpoint selection, corrected array shapes in the variational data and boundary losses, and arclength weighting of ellipse boundary integrals. They are therefore not presented as a rerun of this notebook.

| Method | Relative L² | Relative H¹ | Relative H² |
|---|---:|---:|---:|
| Boundary-adapted | 2.643e-1 ± 3.512e-4 | 3.233e-1 ± 8.386e-4 | 4.031e-1 ± 1.300e-3 |
| Standard | 2.953e-1 ± 1.200e-2 | 4.499e-1 ± 1.393e-2 | 6.530e-1 ± 2.366e-2 |
| Variational/Nitsche | 5.639e-1 ± 1.100e-2 | 1.173 ± 9.637e-2 | 1.197e1 ± 1.703 |
| Over-enforced $d^{3/2}$ | 4.857e-1 ± 3.981e-1 | 5.865e-1 ± 4.187e-1 | 8.943e-1 ± 2.767e-1 |
| Under-enforced $d^{1/2}$ | 9.047e-1 ± 8.524e-3 | 9.202e-1 ± 9.625e-3 | 9.625e-1 ± 8.305e-3 |

The retained training-loss summary from the same runs is:

| Method | PDE or weak loss | Data loss | Boundary loss |
|---|---:|---:|---:|
| Boundary-adapted | 2.316e-4 ± 7.213e-6 | 2.734e-3 ± 2.190e-4 | — |
| Standard | 3.580 ± 5.859e-1 | 9.612e-3 ± 2.935e-3 | 1.466e-11 ± 8.607e-12 |
| Variational/Nitsche | 1.059e-6 ± 6.009e-7 | 1.050e-1 ± 9.377e-3 | 8.653e-18 ± 9.463e-18 |
| Over-enforced $d^{3/2}$ | 6.912 ± 5.865 | 1.565e-1 ± 2.674e-1 | — |
| Under-enforced $d^{1/2}$ | 1.088 ± 2.784e-2 | 4.127e-1 ± 1.489e-2 | — |


## 8. References
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- T. Gustafsson and G. D. McBain, [“scikit-fem: A Python package for finite element assembly”](https://doi.org/10.21105/joss.02369), *JOSS* 5(52), 2369 (2020).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- Y. Liao and P. Ming, [“Deep Nitsche Method”](https://doi.org/10.4208/cicp.OA-2020-0219), *Commun. Comput. Phys.* 29 (2021).
- A. Kendall, Y. Gal, and R. Cipolla, [“Multi-task learning using uncertainty to weigh losses”](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html), CVPR 2018.

- N. Tepakbong, H. Hu, C. Liu, and X. Zhou, “Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds,” ICML 2026. [arXiv preprint](https://arxiv.org/abs/2607.19167).
- N. Tepakbong, H. Hu, C. Liu, and X. Zhou, [“Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds”](https://proceedings.mlr.press/v306/tepakbong26a.html), *Proceedings of the 43rd International Conference on Machine Learning*, PMLR 306, 120120–120175, 2026.
